In [64]:
from google.colab import files
uploaded = files.upload()

import pandas as pd

cols = ['age','workclass','fnlwgt','education','education-num',
        'marital-status','occupation','relationship','race','sex',
        'capital-gain','capital-loss','hours-per-week','native-country','income']

df = pd.read_csv("adult.csv", names=cols, skipinitialspace=True)
df.head()



Saving adult.csv to adult (4).csv


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [65]:
# Replace '?' with NaN
df.replace('?', pd.NA, inplace=True)

# Drop missing values
df.dropna(inplace=True)

# Remove negative values (only numeric columns)
num_cols = df.select_dtypes(include=['int64','float64']).columns
df = df[(df[num_cols] >= 0).all(axis=1)]

print("After Cleaning:", df.shape)

After Cleaning: (30162, 15)


In [66]:
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

df = df[~((df[num_cols] < (Q1 - 1.5 * IQR)) |
          (df[num_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]

print("After Outlier Removal:", df.shape)

After Outlier Removal: (18456, 15)


In [67]:
#Step 1: Fix Target FIRST (IMPORTANT)
df['income'] = df['income'].apply(lambda x: 1 if '>50K' in str(x) else 0)


#Step 2: Encode categorical features
from sklearn.preprocessing import LabelEncoder

for col in df.select_dtypes(include='object').columns:
    df[col] = LabelEncoder().fit_transform(df[col])


#features
X = df.drop('income', axis=1)
y = df['income']


#scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [68]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)


In [69]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", acc_lr)

Logistic Regression Accuracy: 0.8217768147345612


In [70]:
nb = GaussianNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred_nb)

print("Naïve Bayes Accuracy:", acc_nb)

Naïve Bayes Accuracy: 0.797670639219935


In [71]:
print("\nModel Comparison:")
print("Logistic Regression:", acc_lr)
print("Naïve Bayes:", acc_nb)

if acc_lr > acc_nb:
    print("✅ Logistic Regression performs better")
else:
    print("✅ Naïve Bayes performs better")


Model Comparison:
Logistic Regression: 0.8217768147345612
Naïve Bayes: 0.797670639219935
✅ Logistic Regression performs better
